## Greedy Search Decoding

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "gpt2-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

#### Transformers provide generate() for autoregressive models but we will implement this ourself to see what is happening under the hood ..
we'll use "Transformers are the " as input prompts and run decoding for 8 timestaps .

pick max logit at each timestap -> softmax ->get the word

In [2]:
import pandas as pd

input_txt = "Transformers are the "
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
iterations  = []
n_steps = 8
choices_per_step = 5

with torch.no_grad():
    for _ in range(n_steps):
        iteration = dict()
        iteration["input"] = tokenizer.decode(input_ids[0])
        outputs = model(input_ids=input_ids)
        # Select logits of the first batch and the last token and apply softmax
        next_token_logits = outputs.logits[0, -1, :]
        next_token_probs = torch.softmax(next_token_logits, dim=-1)
        sorted_ids = torch.argsort(next_token_probs, dim=-1, descending=True)
        # Store tokens with the highest probabilities
        for choice_idx in range(choices_per_step):
            token_id = sorted_ids[choice_idx]
            token_prob = next_token_probs[token_id].cpu().numpy()
            token_choice = (
                f"{tokenizer.decode(token_id)} ({100 * token_prob:.2f}%)"
            )
            iteration[f"Choice {choice_idx + 1}"] = token_choice
        # Append the predicted next token to the input
        input_ids = torch.cat([input_ids, sorted_ids[None, 0, None]], dim=-1)
        iterations.append(iteration)
pd.DataFrame(iterations)

,input,Choice 1,Choice 2,Choice 3,Choice 4,Choice 5
0,Transformers are the,(15.07%),ills (14.52%),________ (7.28%),icky (4.74%),_____ (4.51%)
1,Transformers are the,most (7.08%),ultimate (4.51%),original (2.48%),""" (1.87%)",main (1.79%)
2,Transformers are the most,popular (17.39%),common (5.30%),powerful (5.28%),famous (4.39%),successful (2.82%)
3,Transformers are the most popular,toy (10.29%),toys (6.68%),Transformers (6.46%),of (5.72%),and (5.04%)
4,Transformers are the most popular toy,line (50.22%),in (9.74%),of (7.60%),lines (5.07%),line (4.08%)
5,Transformers are the most popular toy line,in (36.55%),of (14.28%),", (5.79%)",on (3.88%),. (3.41%)
6,Transformers are the most popular toy line in,the (71.44%),history (7.64%),America (4.29%),Japan (2.48%),all (1.15%)
7,Transformers are the most popular toy line in...,world (68.34%),US (4.86%),history (3.50%),universe (3.18%),United (2.54%)
